# 01. OpenAI API 기초
**SK하이닉스 Autonomous R&D — Day 3** 

> ChatGPT 웹과 **같은 모델**을 코드에서 호출.

---
## 0. 라이브러리 설치

아래 셀을 **최초 1회** 실행.

In [1]:
!pip install openai==1.58.1

   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 2.1/2.1 MB 13.0 MB/s  0:00:00

   ----- ----------------------------------  2/15 [sniffio]
   ------------- --------------------------  5/15 [idna]
   -------------------------- ------------- 10/15 [pydantic]
   -------------------------- ------------- 10/15 [pydantic]
   -------------------------- ------------- 10/15 [pydantic]
   ----------------------------- ---------- 11/15 [httpcore]
   -------------------------------- ------- 12/15 [anyio]
   ---------------------------------- ----- 13/15 [httpx]
   ------------------------------------- -- 14/15 [openai]
   ------------------------------------- -- 14/15 [openai]
   ------------------------------------- -- 14/15 [openai]
   ------------------------------------- -- 14/15 [openai]
   ------------------------------------- -- 14/15 [openai]
   ------------------------------------- -- 14/15 [openai]
   ----------------------

In [2]:
!pip list

Package            Version
------------------ -----------
annotated-types    0.7.0
anyio              4.12.1
asttokens          3.0.0
certifi            2026.6.17
colorama           0.4.6
comm               0.2.3
debugpy            1.8.16
decorator          5.2.1
distro             1.9.0
exceptiongroup     1.3.0
executing          2.2.0
h11                0.16.0
httpcore           1.0.9
httpx              0.28.1
idna               3.18
importlib_metadata 8.7.0
ipykernel          6.30.1
ipython            8.18.1
jedi               0.19.2
jiter              0.15.0
jupyter_client     8.6.3
jupyter_core       5.8.1
matplotlib-inline  0.1.7
nest_asyncio       1.6.0
openai             1.58.1
packaging          26.2
parso              0.8.4
pickleshare        0.7.5
pip                26.0.1
platformdirs       4.3.8
prompt_toolkit     3.0.51
psutil             7.0.0
pure_eval          0.2.3
pydantic           2.13.4
pydantic_core      2.46.4
Pygments           2.19.2
python-dateutil    2.9.0.p

In [ ]:
!pip install openai python-dotenv   # key 관리를 위해

---
## 1. LLM API란?

강의자료 요약:

| 구분 | ChatGPT 웹 | API |
|------|------------|-----|
| 사용자 | 사람이 직접 입력 | **프로그램**이 요청 |
| 결과 | 화면에 표시 | **response 객체**로 반환 |
| 본질 | 대화 | **Prompt → Text** |

API 요청의 핵심 3요소:
1. **`model`** — 어떤 LLM을 쓸지
2. **`messages`** — system / user (대화 내용)
3. **`temperature`** — 답변의 무작위성 (0=일관, 1=창의)

---
## 2. API 키 연결


처음에는 **키를 변수에 넣어** 바로 연결해 봅니다.

> OpenAI Platform → [API keys](https://platform.openai.com/api-keys) 에서 발급

In [ ]:
from openai import OpenAI

# 아래에 본인 키를 붙여넣고 실행
api_key = ''

client = OpenAI(api_key=api_key)

In [5]:
# 연결 테스트
test = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[{'role': 'user', 'content': 'Hi, reply with one word: OK'}],
    max_tokens=10,
)
print('연결 성공:', test.choices[0].message.content)

연결 성공: OK


1. **`.env` 파일**에 키 저장 (코드와 분리) (OPENAI_API_KEY=sk...)
2. **`.gitignore`**에 `.env` 등록 → Git에 **절대 올리지 않음**

In [9]:
from pathlib import Path

root = Path('..')  # ttest/
env_path = root / '.env'
gitignore_path = root / '.gitignore'

print('.env 존재:', env_path.exists(), '→', env_path.resolve())
print('.gitignore 존재:', gitignore_path.exists())

if gitignore_path.exists():
    content = gitignore_path.read_text(encoding='utf-8')
    print('.env in .gitignore:', '.env' in content)

.env 존재: True → C:\Users\Admin\Desktop\AI Autonomous\cursor\.env
.gitignore 존재: True
.env in .gitignore: True


In [10]:
from dotenv import load_dotenv
import os

load_dotenv(root / '.env')
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    raise ValueError('ttest/.env에 OPENAI_API_KEY=sk-... 를 설정하세요.')

client = OpenAI(api_key=api_key)
print('✅ .env에서 API 키 로드 완료 (코드에 키 없음)')

✅ .env에서 API 키 로드 완료 (코드에 키 없음)


---
## 3. 첫 API 호출 

`chat.completions.create()` — **messages** 리스트를 보내면 AI가 답변합니다.

| role | 역할 |
|------|------|
| `system` | AI 역할·규칙 (선택) |
| `user` | 사용자 질문 |

In [ ]:
response = client.chat.completions.create(
    model='gpt-4o-mini',
    temperature=0.1,
    messages=[
        {'role': 'system', 'content': 'You are a helpful assistant.'},  # 시스템
        {'role': 'user',   'content': '2002년 월드컵 우승 팀이 어디야?'},   # 실제 질문
    ],
)

In [17]:
response.choices[0].message.content

'2002년 월드컵 우승 팀은 브라질입니다. 브라질은 결승에서 독일을 2-0으로 이기고 5번째 월드컵 트로피를 차지했습니다.'

In [26]:
####### 2022년 월드컵 우승팀 물어보기

response = client.chat.completions.create(
    model='gpt-4o-mini',
    temperature=0.1,
    messages=[
        {'role': 'system', 'content': 'You are a helpful assistant'},
        {'role': 'user', 'content': '2026 월드컵 한국이랑 멕시코 중에 누가 이겼어?'},
    ]
)



In [27]:
print(response.choices[0].message.content)

2026년 월드컵은 아직 열리지 않았습니다. 따라서 한국과 멕시코의 경기 결과에 대한 정보는 현재로서는 알 수 없습니다. 2026년 월드컵은 미국, 캐나다, 멕시코에서 공동 개최될 예정이며, 경기가 진행된 후에 결과를 확인할 수 있을 것입니다.


---
## 4. 응답(Response) 객체

API는 **텍스트만**이 아니라 **객체**를 돌려줍니다.

| 필드 | 의미 |
|------|------|
| `response.choices[0].message.content` | **실제 답변 텍스트** ← 가장 많이 사용 |
| `response.choices[0].finish_reason` | `stop`=정상 종료, `length`=토큰 초과 |
| `response.usage.total_tokens` | 이번 호출에 쓴 토큰 수 (비용 참고) |
| `response.id` | 요청 ID (로그·디버깅용) |

In [28]:
print('ID:', response.id)
print('finish_reason:', response.choices[0].finish_reason)
print('답변:', response.choices[0].message.content)
print('토큰 사용:', response.usage.total_tokens, '(prompt:', response.usage.prompt_tokens,
      '/ completion:', response.usage.completion_tokens, ')')

ID: chatcmpl-DuBY0MtPAiDutnVHcWPROES79TiFk
finish_reason: stop
답변: 2026년 월드컵은 아직 열리지 않았습니다. 따라서 한국과 멕시코의 경기 결과에 대한 정보는 현재로서는 알 수 없습니다. 2026년 월드컵은 미국, 캐나다, 멕시코에서 공동 개최될 예정이며, 경기가 진행된 후에 결과를 확인할 수 있을 것입니다.
토큰 사용: 108 (prompt: 36 / completion: 72 )


---
## 5. System / User 프롬프트 

같은 `user` 질문이라도 **`system`에 역할·출력 규칙**을 주면 답변 **형식·깊이·관점**이 달라집니다.

아래는 **의도적으로 짧고 모호한 질문**을 사용합니다. system 유무에 따라 차이가 잘 보입니다.

| | system | user |
|---|--------|------|
| 역할 | 모델의 **역할·규칙·형식** | 이번 **질문·작업** |
| 비유 | 직무 기술서 | 오늘 업무 지시 |

In [29]:
# 질문을 짧고 모호하게 → system이 없으면 범용 답변, 있으면 역할·형식에 맞춘 답변
question = 'for문이 뭐야?'

r1 = client.chat.completions.create(
    model='gpt-4o-mini',
    temperature=0.2,
    messages=[{'role': 'user', 'content': question}],
)

print('=== system 없음 (범용 설명, 길어질 수 있음) ===')
print(r1.choices[0].message.content)

=== system 없음 (범용 설명, 길어질 수 있음) ===
`for`문은 프로그래밍에서 반복문 중 하나로, 특정한 작업을 여러 번 반복할 때 사용됩니다. 주로 리스트, 배열, 문자열 등의 요소를 순회하거나, 특정 범위의 숫자를 반복할 때 사용됩니다. 

예를 들어, Python에서 `for`문을 사용하는 기본적인 예시는 다음과 같습니다:

```python
# 리스트의 각 요소를 출력하는 예
fruits = ['사과', '바나나', '체리']
for fruit in fruits:
    print(fruit)
```

위의 코드에서는 `fruits` 리스트의 각 요소를 하나씩 꺼내어 `fruit` 변수에 저장하고, 그 값을 출력합니다.

또 다른 예로, 특정 범위의 숫자를 반복하는 경우는 다음과 같습니다:

```python
# 0부터 4까지의 숫자를 출력하는 예
for i in range(5):
    print(i)
```

이 코드는 0부터 4까지의 숫자를 출력합니다. `range(5)`는 0부터 4까지의 숫자를 생성하는 함수입니다.

`for`문은 다양한 프로그래밍 언어에서 비슷한 형태로 사용되며, 반복 작업을 간편하게 처리할 수 있게 해줍니다.


In [30]:
r2 = client.chat.completions.create(
    model='gpt-4o-mini',
    temperature=0.2,
    messages=[
        {
            'role': 'system',
            'content': '''You are a Python tutor.
Rules:
- Answer in Korean only
- Use exactly 3 lines: [비유 1문장] / [코드 예시 max 3 lines] / [실무 한 줄]
- No long explanation, no bullet lists''',
        },
        {'role': 'user', 'content': question},
    ],
)

print('=== system 있음 (튜터 + 3줄 형식 강제) ===')
print(r2.choices[0].message.content)

=== system 있음 (튜터 + 3줄 형식 강제) ===
for문은 반복문으로, 마치 반복해서 노래를 부르는 것과 같아요.  
```python
for i in range(5):
    print(i)
```
리스트나 범위의 요소를 쉽게 순회할 수 있습니다.


---
## 6. temperature — **일반 예시**

같은 질문, 다른 `temperature` → 결과가 어떻게 달라지는지 확인합니다.

| temperature | 특징 | 용도 |
|-------------|------|------|
| 0 ~ 0.3 | 일관적, 사실 위주 | 분석, 판정, 코드 |
| 0.7 ~ 1.0 | 다양·창의적 | 브레인스토밍, 카피 |

In [31]:
question = '팀 워크숍 아이스브레이킹 아이디어 1가지만.'

for temp in [0.0, 0.7, 1.0]:
    r = client.chat.completions.create(
        model='gpt-4o-mini',
        temperature=temp,
        messages=[{'role': 'user', 'content': question}],
    )
    print(f'[temperature={temp}] {r.choices[0].message.content}')
    print()

[temperature=0.0] "두 진실과 한 거짓" 게임을 제안합니다. 각 팀원이 자신에 대한 두 가지 진실과 하나의 거짓을 말합니다. 다른 팀원들은 어떤 것이 거짓인지 맞춰야 합니다. 이 게임은 서로에 대한 이해를 높이고, 자연스럽게 대화를 유도하는 데 도움이 됩니다. 재미있고 가벼운 분위기를 만들어 팀원 간의 친밀감을 증진시킬 수 있습니다.

[temperature=0.7] "두 진실과 한 거짓" 게임을 제안합니다. 각 팀원이 자신에 대한 두 가지 진실과 하나의 거짓을 이야기합니다. 다른 팀원들은 어떤 것이 거짓인지 맞추는 게임입니다. 이 활동은 팀원들 간의 친밀감을 높이고 서로에 대해 알아가는 재미있는 방법이 됩니다.

[temperature=1.0] "두 진실과 한 거짓" 게임을 추천합니다. 각 팀원이 자신의 두 가지 진실과 한 가지 거짓을 소개하고, 나머지 팀원들이 어떤 것이 거짓인지 맞추는 게임입니다. 이 활동은 팀원들 간의 친밀감을 높이고 서로를 더 잘 알 수 있는 기회를 제공합니다. 동시에 재미있는 이야기와 다양한 경험을 공유하는 좋은 아이스브레이킹 방법이 될 수 있습니다.



---
## 7. `max_tokens` — 출력 길이 제한

답변이 너무 길어지는 것을 막거나, 비용을 줄일 때 사용합니다.

In [32]:
for max_tok in [20, 100]:
    r = client.chat.completions.create(
        model='gpt-4o-mini',
        temperature=0.3,
        max_tokens=max_tok,
        messages=[{'role': 'user', 'content': '대한민국의 역사를 요약해줘.'}],
    )
    print(f'--- max_tokens={max_tok} (finish: {r.choices[0].finish_reason}) ---')
    print(r.choices[0].message.content)
    print()

--- max_tokens=20 (finish: length) ---
대한민국의 역사는 고대부터 현대에 이르기까지 다양한 사건과 변화를 �

--- max_tokens=100 (finish: length) ---
대한민국의 역사는 고대부터 현대에 이르기까지 다양한 사건과 변화를 포함하고 있습니다. 다음은 대한민국 역사의 주요 단계와 사건을 요약한 것입니다.

1. **고대 및 삼국시대 (기원전 2333년 ~ 668년)**:
   - 고조선: 전설에 따르면 단군이 기원전 2333년에 고조선을 세웠다고 전해집니다.
   - 삼국시대:



---
## 8. `ask()` 함수 — 재사용 패턴

같은 API 호출을 함수로 감싸기.

In [33]:
def ask(user_message, system_message='You are a helpful assistant.', temperature=0.2, max_tokens=None):
    """1턴 질의 — 답변 텍스트만 반환"""
    kwargs = dict(
        model='gpt-4o-mini',
        temperature=temperature,
        messages=[
            {'role': 'system', 'content': system_message},
            {'role': 'user',   'content': user_message},
        ],
    )
    if max_tokens is not None:
        kwargs['max_tokens'] = max_tokens
    response = client.chat.completions.create(**kwargs)
    return response.choices[0].message.content


print(ask('서울 3일 여행 코스 추천해줘. bullet 3개만.', max_tokens=150))

서울 3일 여행 코스 추천드립니다:

1. **1일차: 역사와 문화 탐방**
   - 경복궁 방문: 고궁 탐방 및 국립민속박물관 관람
   - 북촌 한옥마을 산책: 전통 한옥과 예쁜 카페 탐방
   - 인사동 거리: 전통 공예품 쇼핑 및 맛있는 전통 음식 즐기기

2. **2일차: 현대 서울 경험하기**
   - 명동: 쇼핑 및 길거리 음식 체험
   - 남산타워(N서울타워): 서울 전경 감상 및 케이블카 이용
   - 홍대: 젊은 문화


---
## 9. 멀티턴 대화 

이전 대화를 `messages`에 **그대로 이어 붙이면** 후속 질문이 가능합니다.

```
system → user → assistant → user → ...
```

### 기존대화

In [34]:
messages = [
    {'role': 'system', 'content': 'You are a helpful travel assistant. Answer in Korean.'},
    {'role': 'user',   'content': '제주도 2박 3일 여행 계획 짜줘. 하루 요약 1줄씩.'},
]

r1 = client.chat.completions.create(model='gpt-4o-mini', temperature=0.3, messages=messages)
answer1 = r1.choices[0].message.content
print('=== 1턴 ===')
print(answer1)

messages = [
    {'role': 'system', 'content': 'You are a helpful travel assistant. Answer in Korean.'},
    {'role': 'user',   'content': '2일차만 더 자세히. 맛집 2곳 포함.'},
]

r2 = client.chat.completions.create(model='gpt-4o-mini', temperature=0.3, messages=messages)
print('\n=== 2턴 (맥락 유지) ===')
print(r2.choices[0].message.content)

=== 1턴 ===
**1일차:** 제주공항 도착 후, 한라산 등반 시작하여 정상에서의 경치를 즐기고, 저녁에는 제주 전통 음식인 흑돼지 구이를 맛본다.

**2일차:** 성산일출봉에서 일출을 감상한 후, 우도 섬으로 배를 타고 이동하여 해변에서 여유로운 시간을 보내고, 저녁에는 해산물 요리를 즐긴다.

**3일차:** 제주 민속촌을 방문하여 제주 전통 문화를 체험하고, 마지막으로 협재 해수욕장에서 수영과 일광욕을 즐긴 후, 제주공항으로 돌아간다.

=== 2턴 (맥락 유지) ===
물론입니다! 2일차 일정을 자세히 설명해드릴게요.

### 2일차 일정

**아침:**
- **식사:** 현지 유명 카페에서 아침 식사. 추천 메뉴는 프렌치 토스트와 커피입니다. 이 카페는 분위기가 좋고, 지역 주민들도 자주 찾는 곳입니다.

**오전:**
- **관광지:** 유명한 박물관 방문. 이 박물관은 지역의 역사와 문화를 잘 보여주는 곳으로, 다양한 전시가 열리고 있습니다. 특히, 특별 전시가 있는 날이면 더욱 흥미롭습니다.

**점심:**
- **맛집 1:** 현지에서 유명한 국수집. 이곳의 대표 메뉴는 손으로 만든 면과 시원한 육수입니다. 국수와 함께 제공되는 다양한 반찬도 맛볼 수 있습니다.

**오후:**
- **관광지:** 아름다운 공원 산책. 이 공원은 자연 경관이 뛰어나고, 산책로가 잘 조성되어 있어 여유롭게 산책하기 좋습니다. 사진 찍기에도 좋은 장소입니다.

**저녁:**
- **맛집 2:** 해산물 전문 식당. 신선한 해산물을 사용한 다양한 요리를 제공합니다. 추천 메뉴는 회와 해물탕입니다. 식사 후에는 근처의 야경 명소를 방문해 보세요.

**밤:**
- **자유 시간:** 지역의 바나 카페에서 여유롭게 시간을 보내며, 현지인들과의 소통을 즐길 수 있습니다.

이렇게 2일차 일정을 구성해보았습니다. 추가적인 정보가 필요하시면 언제든지 말씀해 주세요!


### 멀티턴

### Assistant 메시지는 앞서 있었던 prompt들(user/system)에 대한 모델의 응답을 구성하며, 종종 대화 상태를 유지하기 위해 히스토리의 일부로 저장됩니다.

#### 특징
모델이 생성한 응답을 포함한다.

이전 message의 context 처리를 위해 대화의 연속성 유지를 돕는다.

다음 API 호출에서는 assistant 메시지를 포함시켜야 문맥이 이어짐.

In [35]:
messages = [
    {'role': 'system', 'content': 'You are a helpful travel assistant. Answer in Korean.'},
    {'role': 'user',   'content': '제주도 2박 3일 여행 계획 짜줘. 하루 요약 1줄씩.'},
]

r1 = client.chat.completions.create(model='gpt-4o-mini', temperature=0.3, messages=messages)
answer1 = r1.choices[0].message.content
print('=== 1턴 ===')
print(answer1)



=== 1턴 ===
제주도 2박 3일 여행 계획입니다.

**1일차:** 제주공항 도착 후 한라산 등반 및 성산 일출봉 관람, 저녁에는 제주 전통 음식인 갈치조림 맛보기.

**2일차:** 협재 해수욕장에서 해수욕과 일광욕을 즐기고, 오후에는 오설록 티 뮤지엄에서 차 문화 체험 후, 저녁에는 흑돼지 바비큐.

**3일차:** 만장굴 탐방 후, 우도 섬으로 배를 타고 이동하여 자전거 여행 및 해변 탐방, 마지막으로 제주공항으로 돌아가기.


In [38]:
messages.append({'role': 'assistant', 'content': answer1})
messages.append({'role': 'user', 'content': '2일차만 더 자세히. 맛집 2곳 포함.'})


In [39]:
messages

[{'role': 'system',
  'content': 'You are a helpful travel assistant. Answer in Korean.'},
 {'role': 'user', 'content': '제주도 2박 3일 여행 계획 짜줘. 하루 요약 1줄씩.'},
 {'role': 'assistant',
  'content': '제주도 2박 3일 여행 계획입니다.\n\n**1일차:** 제주공항 도착 후 한라산 등반 및 성산 일출봉 관람, 저녁에는 제주 전통 음식인 갈치조림 맛보기.\n\n**2일차:** 협재 해수욕장에서 해수욕과 일광욕을 즐기고, 오후에는 오설록 티 뮤지엄에서 차 문화 체험 후, 저녁에는 흑돼지 바비큐.\n\n**3일차:** 만장굴 탐방 후, 우도 섬으로 배를 타고 이동하여 자전거 여행 및 해변 탐방, 마지막으로 제주공항으로 돌아가기.'},
 {'role': 'user', 'content': '2일차만 더 자세히. 맛집 2곳 포함.'}]

In [40]:

r2 = client.chat.completions.create(model='gpt-4o-mini', temperature=0.3, messages=messages)
print('\n=== 2턴 (맥락 유지) ===')
print(r2.choices[0].message.content)


=== 2턴 (맥락 유지) ===
2일차 상세 계획입니다.

**오전:** 협재 해수욕장에서 해수욕과 일광욕을 즐기세요. 맑은 바다와 아름다운 경치를 감상하며 여유로운 시간을 보낼 수 있습니다.

**점심:** 협재 해수욕장 근처에 있는 **'협재해수욕장 횟집'**에서 신선한 회와 해산물 요리를 맛보세요. 특히, 제주도에서 유명한 전복회와 회덮밥이 인기입니다.

**오후:** 오설록 티 뮤지엄으로 이동하여 제주 차 문화에 대해 배우고 다양한 차를 시음해 보세요. 아름다운 정원에서 산책도 즐길 수 있습니다.

**저녁:** 제주시로 돌아와 **'돈사돈'**에서 제주 흑돼지 바비큐를 즐기세요. 고소하고 쫄깃한 흑돼지 구이를 맛보며 제주도의 맛을 제대로 느껴보세요.

**저녁 후:** 근처 카페에서 제주 특산물인 한라봉 주스를 마시며 하루를 마무리하세요.


---
## ✏️ 실습 1

아래 주제로 **2턴** 대화를 만들어 보세요.

1턴: "Python for문 기본 문법 설명해줘"
2턴: "방금 설명한 걸로 1~5 합 구하는 코드 예시만 보여줘"

In [ ]:
# ── 여기에 작성 ──
messages = [
    {'role': 'system', 'content': 'You are a Python tutor. Answer in Korean.'},
    # {'role': 'user', 'content': '...'},
]
# r1 = client.chat.completions.create(...)
# messages.append(...)
# r2 = client.chat.completions.create(...)

---
## ✏️ 실습 1

커서 프롬프트 : 3일차 폴더에 스트림릿과 실습코드의 openai api를 이용해서 간단한 챗봇을 만드는 python 코드 만들어줘

In [41]:
!pip install streamlit

  Using cached numpy-2.0.2-cp39-cp39-win_amd64.whl.metadata (59 kB)
   ---------------------------------------- 0.0/10.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.1 MB ? eta -:--:--
   - -------------------------------------- 0.3/10.1 MB ? eta -:--:--
   -- ------------------------------------- 0.5/10.1 MB 5.6 MB/s eta 0:00:02
   ---- ----------------------------------- 1.0/10.1 MB 2.5 MB/s eta 0:00:04
   ------ --------------------------------- 1.6/10.1 MB 2.4 MB/s eta 0:00:04
   ------- -------------------------------- 1.8/10.1 MB 2.0 MB/s eta 0:00:05
   ---------- ----------------------------- 2.6/10.1 MB 2.2 MB/s eta 0:00:04
   ----------- ---------------------------- 2.9/10.1 MB 2.2 MB/s eta 0:00:04
   -------------- ------------------------- 3.7/10.1 MB 2.3 MB/s eta 0:00:03
   --------------- ------------------------ 3.9/10.1 MB 2.2 MB/s eta 0:00:03
   --------------- ------------------------ 3.9/10.1 MB 2.2 MB/s eta 0:00:03
   ---------------- ---------